# LRN Training Example

This notebook trains an `LRNModel` on multilayer SIMNRA datasets discovered under a user-provided root folder.

The workflow is:

1. Recursively scan an absolute dataset root for `.h5`, `.hdf5`, or `.hf5` files.
2. Pad each case into the open-parameter layout of the max-layer dataset.
3. Build an LRN schema from that max-layer input spec.
4. Train the network on one selected output method such as `RBS`.


In [ ]:
from __future__ import annotations

from pathlib import Path
import gc
import sys 
sys.path.append("../")

import matplotlib.pyplot as plt
import numpy as np
import torch

from ibamlkit.data import DatasetBatchReader
from ibamlkit.models.forward import LRNModel, LTNModel, MLPModel, CNNModel, CNN2Model, build_lrn_model_schema
from ibamlkit.pileup import (
    apply_channel_space_pileup,
    compute_rebin_energy_edges,
    convert_energy_spectra_to_channel_space_and_pileup,
    convert_to_channel_space_and_pileup_batch,
    fast_pileup_batch,
    rebin_spectra_to_energy_space,
    rebin_histogram,
    resolve_channel_conversion_arrays,
)
from ibamlkit.training import (
    ConstantFactorTransform,
    EpochSchedule,
    IdentityTransform,
    Chi2Loss,
    LayerwiseConcentrationNormalizer,
    ParameterBoundMinMaxScaler,
    SelectiveMinMaxScaler,
    ModelPackageArtifacts,
    PeakAwareLoss,
    SupervisedTrainer,
    export_as_package,
    TransformPipeline,
    prepare_variable_layer_surrogate_dataset,
    shuffle_in_unison,
    split_train_val_test,
)
from ibamlkit.validation import (
    SIMNRABatchSimulator,
    SimulationBatchResult,
    SurrogateBatchSimulator,
    calculate_chi2_batch,
    fit_open_parameters,
)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

In [ ]:
DATASET_FILE_SUFFIXES = {".h5", ".hdf5", ".hf5"}


def require_absolute_directory(path: Path) -> Path:
    path = Path(path)
    if not path.is_absolute():
        raise ValueError(f"Dataset scan root must be an absolute path, got: {path}")
    if not path.exists():
        raise FileNotFoundError(f"Dataset scan root does not exist: {path}")
    if not path.is_dir():
        raise NotADirectoryError(f"Dataset scan root is not a directory: {path}")
    return path


def collect_dataset_file_groups(scan_root: Path) -> list[tuple[Path, list[Path]]]:
    scan_root = require_absolute_directory(scan_root)
    grouped_paths: dict[Path, list[Path]] = {}
    for path in sorted(scan_root.rglob("*")):
        if not path.is_file() or path.suffix.lower() not in DATASET_FILE_SUFFIXES:
            continue
        grouped_paths.setdefault(path.parent, []).append(path)
    if not grouped_paths:
        raise FileNotFoundError(
            f"No dataset files with suffixes {sorted(DATASET_FILE_SUFFIXES)} were found under {scan_root}"
        )
    return sorted(grouped_paths.items(), key=lambda item: str(item[0]).lower())


def load_case_dataset(paths: list[Path]):
    reader = DatasetBatchReader()
    dataset = reader.load_many(paths)
    return dataset, paths


def print_array_stats(name: str, x: np.ndarray) -> None:
    x = np.asarray(x)
    if x.ndim == 0:
        print(f"{name}: value={float(x):.6g}")
        return
    mean_value = float(np.mean(x, dtype=np.float64))
    min_value = float(np.min(x))
    max_value = float(np.max(x))
    if x.size <= 1_000_000:
        median_value = float(np.median(x))
        print(
            f"{name}: shape={x.shape}, min={min_value:.6g}, max={max_value:.6g}, "
            f"mean={mean_value:.6g}, median={median_value:.6g}"
        )
    else:
        print(
            f"{name}: shape={x.shape}, min={min_value:.6g}, max={max_value:.6g}, "
            f"mean={mean_value:.6g}"
        )


In [ ]:
dataset_root = Path(r"D:/IBAMLKit/examples/datasets/multilayer_7_elements_seed_1")  # Update this path to your dataset locatio
#dataset_root = Path(r"D:/IBAMLKit/examples/datasets/14el")  # Update this path to your dataset locatio
method_name = "RBS"
target_width = 5000 #NRA length 6000, RBS 8400
target_scale_factor = 1e-6
energy_bin_width = 2.5
energy_spectrum_scale = 1.0
seed = 7
val_count = 5000
test_count = 5000

dataset_root = require_absolute_directory(dataset_root)
case_groups = collect_dataset_file_groups(dataset_root)
datasets = []
for case_dir, paths in case_groups:
    try:
        dataset, paths = load_case_dataset(paths)
        n_layers = int(dataset.input_spec.generation_info.get("n_layers", 0))
        print(
            f"Loaded case {case_dir}: {len(paths)} file(s), "
            f"samples={dataset.sample_count}, n_layers={n_layers}"
        )
        datasets.append(dataset)
    except Exception as e:
        print(f"Error loading case {case_dir}: {e}")

if not datasets:
    raise RuntimeError(f"No datasets could be loaded from {dataset_root}")

reference_dataset = max(
    datasets,
    key=lambda dataset: int(dataset.input_spec.generation_info.get("n_layers", 0)),
)
bootstrap_schema = build_lrn_model_schema(
    reference_dataset.input_spec,
    model_name=f"lrn_{method_name.lower()}",
    task_method_names=[method_name],
    output_spectra_lengths={method_name: 1},
)
prepared = prepare_variable_layer_surrogate_dataset(
    datasets,
    schema=bootstrap_schema,
    method_name=method_name,
)
reference_dataset = prepared.reference_dataset
reference_open_parameter_names = [parameter.name for parameter in reference_dataset.input_spec.open_parameters]
full_open_parameter_values = np.asarray(prepared.inputs_full, dtype=np.float32)
channel_targets_raw = np.asarray(prepared.targets, dtype=np.float32)
channel_target_lengths = prepared.target_lengths
if channel_target_lengths is not None:
    channel_target_lengths = np.asarray(channel_target_lengths, dtype=np.int32)
    valid_mask = channel_target_lengths > 0
    dropped_count = int((~valid_mask).sum())
    if dropped_count:
        full_open_parameter_values = np.asarray(full_open_parameter_values[valid_mask], dtype=np.float32)
        channel_targets_raw = np.asarray(channel_targets_raw[valid_mask], dtype=np.float32)
        channel_target_lengths = np.asarray(channel_target_lengths[valid_mask], dtype=np.int32)
        prepared_inputs_selected = np.asarray(prepared.inputs_selected[valid_mask], dtype=np.float32)
        print(f"Discarded {dropped_count} invalid samples with zero-length {method_name} spectra.")
    else:
        prepared_inputs_selected = np.asarray(prepared.inputs_selected, dtype=np.float32)
else:
    prepared_inputs_selected = np.asarray(prepared.inputs_selected, dtype=np.float32)

sample_count = int(prepared_inputs_selected.shape[0])
permutation = np.random.default_rng(seed).permutation(sample_count)
prepared_inputs_selected = np.asarray(prepared_inputs_selected[permutation], dtype=np.float32)
full_open_parameter_values = np.asarray(full_open_parameter_values[permutation], dtype=np.float32)

calibration_offset_all, calibration_linear_all, calibration_quadratic_all, normalization_factor_all = resolve_channel_conversion_arrays(
    reference_dataset.input_spec,
    reference_open_parameter_names,
    full_open_parameter_values,
    method_name=method_name,
    energy_spectrum_scale=energy_spectrum_scale,
)

energy_bin_edges = compute_rebin_energy_edges(
    channel_targets_raw,
    channel_target_lengths,
    calibration_offset=calibration_offset_all,
    calibration_linear=calibration_linear_all,
    calibration_quadratic=calibration_quadratic_all,
    energy_bin_width=energy_bin_width,
    row_indices=permutation,
    show_progress=True,
    progress_desc=f"Rebinning {method_name}",
)

energy_output_width = min(target_width, max(energy_bin_edges.shape[0] - 1, 0))
artifact_dir = Path.cwd() / "artifacts"
artifact_dir.mkdir(parents=True, exist_ok=True)
energy_targets_path = artifact_dir / f"{method_name.lower()}_energy_targets.dat"
energy_targets_raw = np.memmap(
    energy_targets_path,
    mode="w+",
    dtype=np.float32,
    shape=(sample_count, energy_output_width),
)

energy_target_lengths = np.zeros((sample_count,), dtype=np.int32)
rebin_spectra_to_energy_space(
    channel_targets_raw,
    channel_target_lengths,
    calibration_offset=calibration_offset_all,
    calibration_linear=calibration_linear_all,
    calibration_quadratic=calibration_quadratic_all,
    energy_bin_width=energy_bin_width,
    target_width=target_width,
    energy_spectrum_scale=target_scale_factor,
    row_indices=permutation,
    energy_edges=energy_bin_edges,
    out=energy_targets_raw,
    show_progress=True,
    progress_desc=f"Rebinning {method_name}",
)

schema = build_lrn_model_schema(
    reference_dataset.input_spec,
    model_name=f"lrn_{method_name.lower()}_e_space",
    task_method_names=[method_name],
    output_spectra_lengths={method_name: int(energy_targets_raw.shape[1])},
)
x = prepared_inputs_selected
y = energy_targets_raw
y_lengths = energy_target_lengths

output_transform = ConstantFactorTransform(target_scale_factor)
output_transform.fit(np.zeros((1, y.shape[1]), dtype=np.float32))

print(f"Raw channel-space targets released; E-space targets shape={y.shape}, dtype={y.dtype}")

del prepared
del channel_targets_raw
gc.collect()

print("Dataset root:", dataset_root)
print("Reference layer count:", reference_dataset.input_spec.generation_info.get("n_layers"))
print("Input matrix shape:", x.shape)
print("E-space output matrix shape:", y.shape)
print("Energy bin width:", energy_bin_width)
print("Energy bin edge count:", energy_bin_edges.shape[0])
print("Schema input dimension:", schema.inputs.dimension)
print("Schema E-space output width:", schema.outputs.spectra_lengths[method_name])

In [ ]:
split = split_train_val_test(
    x,
    y,
    val_count=val_count,
    test_count=test_count,
    target_lengths=y_lengths,
)

x_train = split.train_inputs
y_train = split.train_targets
x_val = split.val_inputs
y_val = split.val_targets
x_test = split.test_inputs
y_test = split.test_targets
test_lengths = split.test_target_lengths

train_count = x_train.shape[0]
val_end = train_count + x_val.shape[0]
full_open_parameter_values_train = full_open_parameter_values[:train_count]
full_open_parameter_values_val = full_open_parameter_values[train_count:val_end]
full_open_parameter_values_test = full_open_parameter_values[val_end:]
concentration_normalizer = LayerwiseConcentrationNormalizer.from_schema(schema)
feature_scaler = ParameterBoundMinMaxScaler.from_parameters(
    reference_dataset.input_spec.open_parameters,
    passthrough_kinds=("concentration",),
    fixed_bounds_by_kind={"thickness": (0.0, 100000.0)},
    low=0.0,
    high=1.0,
)
input_scaler = TransformPipeline([
    concentration_normalizer,
    feature_scaler,
])
input_scaler.fit(x_train)
x_scaled_path = artifact_dir / f"{method_name.lower()}_input_scaled.dat"
x_scaled = np.memmap(
    x_scaled_path,
    mode="w+",
    dtype=np.float32,
    shape=x.shape,
)
scale_chunk_size = 50000
for start in range(0, x.shape[0], scale_chunk_size):
    end = min(start + scale_chunk_size, x.shape[0])
    x_scaled[start:end] = input_scaler.transform(x[start:end])
x_train_scaled = x_scaled[:train_count]
x_val_scaled = x_scaled[train_count:val_end]
x_test_scaled = x_scaled[val_end:]
del x_train, x_val, x
gc.collect()

print("Train:", x_train_scaled.shape, y_train.shape)
print("Val:", x_val_scaled.shape, y_val.shape)
print("Test:", x_test_scaled.shape, y_test.shape)
print(
    "Input transform:",
    "layerwise concentration normalization + parameter-bound min-max scaling",
    f"pass-through concentration columns={(~feature_scaler.scale_columns).sum()}",
    f"fixed-range columns={feature_scaler.fixed_range_columns.sum()}",
    f"scaled columns={feature_scaler.scale_columns.sum()}",
)
print("Channel-space test targets: omitted to avoid duplicating the raw matrix")

In [ ]:
dataset.input_spec.open_parameters

In [ ]:
x_train_scaled [2]

In [ ]:
for i in range(3) : 
    plt.plot(y_train[i], label="Train")


In [ ]:

ltn_model = LTNModel(
    schema,
    model_dim=256,
    num_heads=1,
    num_encoder_layers=2,
    feedforward_dim=512,
    dropout=0.1,
    decoder_hidden_sizes=(512, 512),
    refiner_hidden_channels=32,
    refiner_kernel_size=17,
)
    
print("LTN Model created")
print(f"  Input dimension: {ltn_model.input_dimension}")
print(f"  Output dimension: {ltn_model.output_feature_dimension}")
print(f"  Total parameters: {sum(p.numel() for p in ltn_model.parameters()):,}")

In [ ]:
lrn_model = LRNModel(
    schema,
    hidden_size= 256,
    contribution_size= 512,
    setup_embedding_dim = 32,
    layer_embedding_dim = 256,
    block_hidden_sizes = (768, 768),
    decoder_hidden_sizes = (768, 768),
    refiner_hidden_channels=32,
    refiner_kernel_size=8,
)

print("LRN Model created")
print(f"  Input dimension: {lrn_model.input_dimension}")
print(f"  Output dimension: {lrn_model.output_feature_dimension}")
print(f"  Total parameters: {sum(p.numel() for p in lrn_model.parameters()):,}")

## MLP Model

Pure feedforward multi-layer perceptron model as an alternative architecture.

In [ ]:
mlp_model = MLPModel(
    schema,
    hidden_sizes=(768, 768, 512),
    dropout_rate=0.1,
    refiner_hidden_channels=32,
    refiner_kernel_size=16,
)

print("MLP Model created")
print(f"  Input dimension: {mlp_model.input_dimension}")
print(f"  Output dimension: {mlp_model.output_feature_dimension}")
print(f"  Total parameters: {sum(p.numel() for p in mlp_model.parameters()):,}")

## CNN Model

Convolutional neural network model treating inputs as 1D sequences.

In [ ]:
cnn_model = CNNModel(
    schema,
    initial_channels=32,
    num_conv_layers=3,
    kernel_size=5,
    decoder_hidden_sizes=(512, 512),
    dropout_rate=0.1,
    refiner_hidden_channels=32,
    refiner_kernel_size=16,
)

print("CNN Model created")
print(f"  Input dimension: {cnn_model.input_dimension}")
print(f"  Output dimension: {cnn_model.output_feature_dimension}")
print(f"  Total parameters: {sum(p.numel() for p in cnn_model.parameters()):,}")

## CNN2 Model

Alternative CNN architecture with dense projection and reshape.

In [ ]:
cnn2_model = CNN2Model(
    schema,
    dense_size=4096,
    channels_1=256,
    channels_2=512,
    bottleneck_length=32,
    kernel_size=3,
    decoder_hidden_sizes=(),
    dropout_rate=0.1,
    refiner_hidden_channels=32,
    refiner_kernel_size=16,
)

print("CNN2 Model created")
print(f"  Input dimension: {cnn2_model.input_dimension}")
print(f"  Output dimension: {cnn2_model.output_feature_dimension}")
print(f"  Total parameters: {sum(p.numel() for p in cnn2_model.parameters()):,}")

## Train LRN Model

In [ ]:
lrn_trainer = SupervisedTrainer(
    device=device,
    loss_fn=Chi2Loss(),
    optimizer_name="adamw",
    weight_decay=0.001,
    max_grad_norm=5.0,
    early_stopping_patience=50,
    track_train_loss=False,
    batch_shuffle_mode="full",
    shuffle_block_size=65536,
    verbose=True,
    log_every_epochs=1,
)

lrn_result = lrn_trainer.fit(
    lrn_model,
    train_inputs=x_train_scaled,
    train_targets=y_train,
    val_inputs=x_val_scaled,
    val_targets=y_val,
    schedule=[
        EpochSchedule(learning_rate=1e-3, epochs=20, batch_size=1024),
        EpochSchedule(learning_rate=5e-4, epochs=20, batch_size=1024),
        EpochSchedule(learning_rate=1e-4, epochs=20, batch_size=1024),
        EpochSchedule(learning_rate=5e-5, epochs=5, batch_size=1024),
    ],
)

print("LRN Training completed:")
print(lrn_result)

In [ ]:
ltn_trainer = SupervisedTrainer(
    device=device,
    loss_fn=Chi2Loss(),
    optimizer_name="adamw",
    weight_decay=1e-3,
    max_grad_norm=5.0,
    early_stopping_patience=50,
    track_train_loss=False,
    batch_shuffle_mode="full",
    shuffle_block_size=65536,
    verbose=True,
    log_every_epochs=1,
)

ltn_result = ltn_trainer.fit(
    ltn_model,
    train_inputs=x_train_scaled,
    train_targets=y_train,
    val_inputs=x_val_scaled,
    val_targets=y_val,
    schedule=[
        #EpochSchedule(learning_rate=1e-3, epochs=20, batch_size=1024),
        #EpochSchedule(learning_rate=5e-4, epochs=20, batch_size=1024),
        EpochSchedule(learning_rate=1e-4, epochs=20, batch_size=1024),
        EpochSchedule(learning_rate=5e-5, epochs=5, batch_size=1024),
    ],
)

print("LTN Training completed:")
print(ltn_result)

## Train MLP Model

Train the MLP surrogate model with the same dataset.

In [ ]:
mlp_trainer = SupervisedTrainer(
    device=device,
    loss_fn=Chi2Loss(),
    optimizer_name="adamw",
    weight_decay=1e-3,
    max_grad_norm=5.0,
    early_stopping_patience=50,
    track_train_loss=False,
    batch_shuffle_mode="full",
    shuffle_block_size=65536,
    verbose=True,
    log_every_epochs=1,
)

mlp_result = mlp_trainer.fit(
    mlp_model,
    train_inputs=x_train_scaled,
    train_targets=y_train,
    val_inputs=x_val_scaled,
    val_targets=y_val,
    schedule=[
        EpochSchedule(learning_rate=1e-3, epochs=20, batch_size=1024),
        EpochSchedule(learning_rate=5e-4, epochs=20, batch_size=1024),
        EpochSchedule(learning_rate=1e-4, epochs=20, batch_size=1024),
        EpochSchedule(learning_rate=5e-5, epochs=5, batch_size=1024),
    ],
)

print("MLP Training completed:")
print(mlp_result)

## Train CNN2 Model

Train the second CNN surrogate model with a dense reshape backbone.

In [ ]:
cnn2_trainer = SupervisedTrainer(
    device=device,
    loss_fn=Chi2Loss(),
    optimizer_name="adamw",
    weight_decay=1e-3,
    max_grad_norm=5.0,
    early_stopping_patience=50,
    track_train_loss=False,
    batch_shuffle_mode="full",
    shuffle_block_size=65536,
    verbose=True,
    log_every_epochs=1,
)

cnn2_result = cnn2_trainer.fit(
    cnn2_model,
    train_inputs=x_train_scaled,
    train_targets=y_train,
    val_inputs=x_val_scaled,
    val_targets=y_val,
    schedule=[
        EpochSchedule(learning_rate=1e-3, epochs=20, batch_size=1024),
        EpochSchedule(learning_rate=5e-4, epochs=20, batch_size=1024),
        EpochSchedule(learning_rate=1e-4, epochs=20, batch_size=1024),
        EpochSchedule(learning_rate=5e-5, epochs=5, batch_size=1024),
    ],
)

print("CNN2 Training completed:")
print(cnn2_result)

## Train CNN Model

Train the CNN surrogate model with the same dataset.

In [ ]:
cnn_trainer = SupervisedTrainer(
    device=device,
    loss_fn=Chi2Loss(),
    optimizer_name="adamw",
    weight_decay=1e-3,
    max_grad_norm=5.0,
    early_stopping_patience=50,
    track_train_loss=False,
    batch_shuffle_mode="full",
    shuffle_block_size=65536,
    verbose=True,
    log_every_epochs=1,
)

cnn_result = cnn_trainer.fit(
    cnn_model,
    train_inputs=x_train_scaled,
    train_targets=y_train,
    val_inputs=x_val_scaled,
    val_targets=y_val,
    schedule=[
        EpochSchedule(learning_rate=1e-3, epochs=20, batch_size=1024),
        EpochSchedule(learning_rate=5e-4, epochs=20, batch_size=1024),
        EpochSchedule(learning_rate=1e-4, epochs=20, batch_size=1024),
        EpochSchedule(learning_rate=5e-5, epochs=5, batch_size=1024),
    ],
)

print("CNN Training completed:")
print(cnn_result)

## Model Comparison

Compare the performance of all three models (LRN, MLP, CNN) on test spectra.

In [ ]:
pred_test_transformed = lrn_model.predict(x_test_scaled).cpu().numpy()
pred_test_e = np.empty_like(pred_test_transformed, dtype=np.float32)
output_transform.inverse_transform(pred_test_transformed, out=pred_test_e)
y_test_e = np.empty_like(y_test, dtype=np.float32)
output_transform.inverse_transform(y_test, out=y_test_e)
del pred_test_transformed

print(f"Predicted E-space test spectra: shape={pred_test_e.shape}, dtype={pred_test_e.dtype}")
chi2_e = np.mean((pred_test_e - y_test_e) ** 2 / (y_test_e + 1.0), axis=1)
print("E-space test mean chi2:", float(np.mean(chi2_e)))
print("E-space test median chi2:", float(np.median(chi2_e)))

n_plot = min(50, x_test_scaled.shape[0])
n_cols = 2
n_rows = int(np.ceil(n_plot / n_cols))

fig, axes = plt.subplots(
    n_rows,
    n_cols,
    figsize=(10, 2.5 * n_rows),
    sharex=False,
    squeeze=True,
)

axes_flat = axes.ravel()

for row_index in range(n_plot):
    ax = axes_flat[row_index]
    e_length = y_test_e.shape[1]
    ax.plot(y_test_e[row_index, :e_length], label="target")
    ax.plot(pred_test_e[row_index, :e_length], linestyle="--", label="prediction")
    ax.set_ylabel(f"sample {row_index}")
    if row_index == 0:
        ax.legend()

    ax.set_xlabel("energy bin")

for ax in axes_flat[n_plot:]:
    ax.set_visible(False)

for ax in axes[-1, :]:
    if ax.get_visible(): pass


fig.subplots_adjust(wspace=0.2, hspace=0.25)
plt.tight_layout()
plt.savefig(artifact_dir / f"{method_name.lower()}_e_space_predictions_7el.pdf", bbox_inches="tight")

In [ ]:
# Evaluate all four models on the test set
print("=" * 60)
print("TEST SET EVALUATION")
print("=" * 60)

for m in [lrn_model, mlp_model, cnn_model, cnn2_model]:
    m.eval()

with torch.no_grad():
    pred_lrn = lrn_model.predict(x_test_scaled).cpu().numpy()
    pred_mlp = mlp_model.predict(x_test_scaled).cpu().numpy()
    pred_cnn = cnn_model.predict(x_test_scaled).cpu().numpy()
    pred_cnn2 = cnn2_model.predict(x_test_scaled).cpu().numpy()

pred_lrn_e = np.empty_like(pred_lrn, dtype=np.float32)
pred_mlp_e = np.empty_like(pred_mlp, dtype=np.float32)
pred_cnn_e = np.empty_like(pred_cnn, dtype=np.float32)
pred_cnn2_e = np.empty_like(pred_cnn2, dtype=np.float32)

output_transform.inverse_transform(pred_lrn, out=pred_lrn_e)
output_transform.inverse_transform(pred_mlp, out=pred_mlp_e)
output_transform.inverse_transform(pred_cnn, out=pred_cnn_e)
output_transform.inverse_transform(pred_cnn2, out=pred_cnn2_e)

chi2_lrn = np.mean((pred_lrn_e - y_test) ** 2 / (y_test + 1.0), axis=1)
chi2_mlp = np.mean((pred_mlp_e - y_test) ** 2 / (y_test + 1.0), axis=1)
chi2_cnn = np.mean((pred_cnn_e - y_test) ** 2 / (y_test + 1.0), axis=1)
chi2_cnn2 = np.mean((pred_cnn2_e - y_test) ** 2 / (y_test + 1.0), axis=1)

print("\nLRN Model:")
print(f"  Mean chi2: {float(np.mean(chi2_lrn)):.6g}")
print(f"  Median chi2: {float(np.median(chi2_lrn)):.6g}")
print(f"  Std dev chi2: {float(np.std(chi2_lrn)):.6g}")

print("\nMLP Model:")
print(f"  Mean chi2: {float(np.mean(chi2_mlp)):.6g}")
print(f"  Median chi2: {float(np.median(chi2_mlp)):.6g}")
print(f"  Std dev chi2: {float(np.std(chi2_mlp)):.6g}")

print("\nCNN Model:")
print(f"  Mean chi2: {float(np.mean(chi2_cnn)):.6g}")
print(f"  Median chi2: {float(np.median(chi2_cnn)):.6g}")
print(f"  Std dev chi2: {float(np.std(chi2_cnn)):.6g}")

print("\nCNN2 Model:")
print(f"  Mean chi2: {float(np.mean(chi2_cnn2)):.6g}")
print(f"  Median chi2: {float(np.median(chi2_cnn2)):.6g}")
print(f"  Std dev chi2: {float(np.std(chi2_cnn2)):.6g}")

n_compare = min(20, x_test_scaled.shape[0])
fig, axes = plt.subplots(n_compare, 1, figsize=(12, 1.2 * n_compare))
if n_compare == 1:
    axes = np.asarray([axes])

for row_idx in range(n_compare):
    e_length = y_test.shape[1]
    axes[row_idx].plot(y_test[row_idx, :e_length], "k-", linewidth=2, label="target")
    axes[row_idx].plot(pred_lrn_e[row_idx, :e_length], "r--", alpha=0.7, label="LRN")
    axes[row_idx].plot(pred_mlp_e[row_idx, :e_length], "g--", alpha=0.7, label="MLP")
    axes[row_idx].plot(pred_cnn_e[row_idx, :e_length], "b--", alpha=0.7, label="CNN")
    axes[row_idx].plot(pred_cnn2_e[row_idx, :e_length], "m--", alpha=0.7, label="CNN2")
    axes[row_idx].set_ylabel(f"sample {row_idx}")
    if row_idx == 0:
        axes[row_idx].legend(loc="upper right")

axes[-1].set_xlabel("energy bin")
fig.suptitle("Model Predictions Comparison")
plt.tight_layout()
plt.show()


## Save MLP and CNN Models

In [ ]:
artifact_dir = Path.cwd() / "artifacts"
artifact_dir.mkdir(parents=True, exist_ok=True)

# Save MLP model
mlp_artifact_path = artifact_dir / f"mlp_{method_name.lower()}_state_dict.pt"
torch.save(
    {
        "model_state_dict": mlp_model.state_dict(),
        "schema": schema,
        "input_scaler": input_scaler,
        "output_transform": output_transform,
        "training_result": mlp_result,
    },
    mlp_artifact_path,
)
print("Saved MLP model:", mlp_artifact_path)

# Save CNN model
cnn_artifact_path = artifact_dir / f"cnn_{method_name.lower()}_state_dict.pt"
torch.save(
    {
        "model_state_dict": cnn_model.state_dict(),
        "schema": schema,
        "input_scaler": input_scaler,
        "output_transform": output_transform,
        "training_result": cnn_result,
    },
    cnn_artifact_path,
)
print("Saved CNN model:", cnn_artifact_path)

# Save CNN2 model
cnn2_artifact_path = artifact_dir / f"cnn2_{method_name.lower()}_state_dict.pt"
torch.save(
    {
        "model_state_dict": cnn2_model.state_dict(),
        "schema": schema,
        "input_scaler": input_scaler,
        "output_transform": output_transform,
        "training_result": cnn2_result,
    },
    cnn2_artifact_path,
)
print("Saved CNN2 model:", cnn2_artifact_path)

# Optional: Save as packages
try:
    mlp_package_path = export_as_package(
        mlp_model,
        ModelPackageArtifacts(
            input_spec=reference_dataset.input_spec,
            input_scaler=input_scaler,
            output_transform=output_transform,
            metadata={
                "method_name": method_name,
                "energy_bin_width": energy_bin_width,
                "energy_output_width": int(y.shape[1]),
                "model_type": "mlp",
            },
        ),
        artifact_dir / f"mlp_{method_name.lower()}_package_7el_1.5_v1.0.zip",
    )
    print("Exported MLP package:", mlp_package_path)
except Exception as e:
    print(f"Could not export MLP package: {e}")

try:
    cnn_package_path = export_as_package(
        cnn_model,
        ModelPackageArtifacts(
            input_spec=reference_dataset.input_spec,
            input_scaler=input_scaler,
            output_transform=output_transform,
            metadata={
                "method_name": method_name,
                "energy_bin_width": energy_bin_width,
                "energy_output_width": int(y.shape[1]),
                "model_type": "cnn",
            },
        ),
        artifact_dir / f"cnn_{method_name.lower()}_package_7el_1.5_v1.0.zip",
    )
    print("Exported CNN package:", cnn_package_path)
except Exception as e:
    print(f"Could not export CNN package: {e}")

try:
    cnn2_package_path = export_as_package(
        cnn2_model,
        ModelPackageArtifacts(
            input_spec=reference_dataset.input_spec,
            input_scaler=input_scaler,
            output_transform=output_transform,
            metadata={
                "method_name": method_name,
                "energy_bin_width": energy_bin_width,
                "energy_output_width": int(y.shape[1]),
                "model_type": "cnn2",
            },
        ),
        artifact_dir / f"cnn2_{method_name.lower()}_package_7el_1.5_v1.0.zip",
    )
    print("Exported CNN2 package:", cnn2_package_path)
except Exception as e:
    print(f"Could not export CNN2 package: {e}")


In [ ]:
package_path = export_as_package(
    lrn_model,
    ModelPackageArtifacts(
        input_spec=reference_dataset.input_spec,
        input_scaler=input_scaler,
        output_transform=output_transform,
        metadata={
            "method_name": method_name,
            "energy_bin_width": energy_bin_width,
            "energy_output_width": int(y.shape[1]),
        },
    ),
    artifact_dir / f"lrn_{method_name.lower()}_package_14el_2mil_v1.0.zip",
)
print("Exported package:", package_path)



In [ ]:
artifact_dir = Path.cwd() / "artifacts"
artifact_dir.mkdir(parents=True, exist_ok=True)
artifact_path = artifact_dir / f"mlp_{method_name.lower()}_state_dict.pt"
torch.save(
    {
        "model_state_dict": mlp_model.state_dict(),
        "schema": schema,
        "input_scaler": input_scaler,
        "output_transform": output_transform,
       # "training_result": result,
    },
    artifact_path,
)
print("Saved:", artifact_path)


In [ ]:
artifact_dir = Path.cwd() / "artifacts"
artifact_dir.mkdir(parents=True, exist_ok=True)
artifact_path = artifact_dir / f"cnn_{method_name.lower()}_state_dict.pt"
torch.save(
    {
        "model_state_dict": cnn_model.state_dict(),
        "schema": schema,
        "input_scaler": input_scaler,
        "output_transform": output_transform,
       # "training_result": result,
    },
    artifact_path,
)
print("Saved:", artifact_path)


In [ ]:
artifact_dir = Path.cwd() / "artifacts"
artifact_dir.mkdir(parents=True, exist_ok=True)
artifact_path = artifact_dir / f"ltn_{method_name.lower()}_state_dict.pt"
torch.save(
    {
        "model_state_dict": ltn_model.state_dict(),
        "schema": schema,
        "input_scaler": input_scaler,
        "output_transform": output_transform,
       # "training_result": result,
    },
    artifact_path,
)
print("Saved:", artifact_path)


In [ ]:
artifact_dir = Path.cwd() / "artifacts"
artifact_dir.mkdir(parents=True, exist_ok=True)
artifact_path = artifact_dir / f"lrn_{method_name.lower()}_state_dict.pt"
torch.save(
    {
        "model_state_dict": lrn_model.state_dict(),
        "schema": schema,
        "input_scaler": input_scaler,
        "output_transform": output_transform,
       # "training_result": result,
    },
    artifact_path,
)
print("Saved:", artifact_path)


In [ ]:
model = LRNModel(
    schema,
    hidden_size= 512,
    contribution_size= 512,
    setup_embedding_dim = 32,
    layer_embedding_dim = 512,
    block_hidden_sizes = (768, 768),
    decoder_hidden_sizes = (768, 768),
    refiner_hidden_channels=32,
    refiner_kernel_size=32,
)
artifact_path = artifact_dir / f"lrn_{method_name.lower()}_state_dict.pt"
model.load_state_dict(torch.load(artifact_path, map_location=device)["model_state_dict"])

In [ ]:
pred_test_transformed = lrn_model.predict(x_test_scaled).cpu().numpy()
pred_test_e = np.empty_like(pred_test_transformed, dtype=np.float32)
output_transform.inverse_transform(pred_test_transformed, out=pred_test_e)
y_test_e = np.empty_like(y_test, dtype=np.float32)
output_transform.inverse_transform(y_test, out=y_test_e)
del pred_test_transformed

print(f"Predicted E-space test spectra: shape={pred_test_e.shape}, dtype={pred_test_e.dtype}")
chi2_e = np.mean((pred_test_e - y_test_e) ** 2 / (y_test_e + 1.0), axis=1)
print("E-space test mean chi2:", float(np.mean(chi2_e)))
print("E-space test median chi2:", float(np.median(chi2_e)))

n_plot = min(50, x_test_scaled.shape[0])
fig, axes = plt.subplots(n_plot, 1, figsize=(12, 1.5 * n_plot), sharex=False)
if n_plot == 1:
    axes = np.asarray([axes])

for row_index in range(n_plot):
    e_length = y_test_e.shape[1]
    ind = row_index
    axes[row_index].plot(y_test_e[ind, :e_length], label="target")
    axes[row_index].plot(pred_test_e[ind, :e_length], linestyle="--", label="prediction")
    axes[row_index].set_ylabel(f"sample {row_index}")
    axes[row_index].set_title("E-space")
    if row_index == 0:
        axes[row_index].legend()

axes[-1].set_xlabel("energy bin")
plt.tight_layout()
plt.show()

In [ ]:
artifact_dir = Path.cwd() / "artifacts"
artifact_dir.mkdir(parents=True, exist_ok=True)
artifact_path = artifact_dir / f"lrn_{method_name.lower()}_state_dict_lrnscaler_7el.pt"
torch.save(
    {
        "model_state_dict": lrn_model.state_dict(),
        "schema": schema,
        "input_scaler": input_scaler,
        "output_transform": output_transform,
        #"training_result": result,
    },
    artifact_path,
)
print("Saved:", artifact_path)

## Parameter Fitting On Test Spectra

The cells below fit the open parameters for a small subset of the test spectra using both the trained surrogate model and SIMNRA.

Notes:
- Surrogate fitting ignores `nthreads` because model inference is already vectorized.
- SIMNRA fitting can be substantially slower; keep `fit_sample_count` small unless you want a long run.


In [ ]:
fit_sample_count = 4
simnra_fit_threads = 4
surrogate_fit_algo = "L_BFGS_B"
simnra_fit_algo = "ARRDE"
surrogate_fit_maxevals = 800
simnra_fit_maxevals = 3000
fit_seed = 123

fit_sample_count = min(fit_sample_count, x_test.shape[0])
x_fit_true = np.asarray(x_test[:fit_sample_count], dtype=np.float32)
observed_channel_pileup = np.asarray(y_test_channel_pileup[:fit_sample_count], dtype=np.float32)
observed_spectra = {method_name: observed_channel_pileup}
observed_lengths = {method_name: np.full((fit_sample_count,), observed_channel_pileup.shape[1], dtype=np.int32)}

open_parameters = list(reference_dataset.input_spec.open_parameters)
lower_bounds = np.asarray([
    0.0 if parameter.lower_bound is None else parameter.lower_bound
    for parameter in open_parameters
], dtype=np.float32)
upper_bounds = np.asarray([
    1.0 if parameter.upper_bound is None else parameter.upper_bound
    for parameter in open_parameters
], dtype=np.float32)
rng = np.random.default_rng(fit_seed)
initial_guess = lower_bounds + rng.uniform(size=x_fit_true.shape).astype(np.float32) * (upper_bounds - lower_bounds)

class SurrogateChannelPileupSimulator:
    def __init__(self, input_spec, schema, model, input_transform, output_inverse_transform, method_name, real_time, live_time, fudge_factor, energy_spectrum_scale):
        self.input_spec = input_spec
        self.schema = schema
        self.model = model
        self.input_transform = input_transform
        self.output_inverse_transform = output_inverse_transform
        self.method_name = method_name
        self.real_time = float(real_time)
        self.live_time = float(live_time)
        self.fudge_factor = float(fudge_factor)
        self.energy_spectrum_scale = float(energy_spectrum_scale)
        self.open_parameter_names = [parameter.name for parameter in input_spec.open_parameters]
        self.base = SurrogateBatchSimulator(
            input_spec=input_spec,
            schema=schema,
            model=model,
            input_transform=input_transform,
            output_inverse_transform=output_inverse_transform,
        )
        self.method_names = [method_name]

    def simulate_batch(self, open_parameter_values, *, fixed_parameter_overrides=None, nthreads=1):
        del nthreads
        energy_result = self.base.simulate_batch(
            open_parameter_values,
            fixed_parameter_overrides=fixed_parameter_overrides,
        )
        a, b, c, r = resolve_channel_conversion_arrays(
            self.input_spec,
            self.open_parameter_names,
            np.asarray(open_parameter_values, dtype=np.float32),
            method_name=self.method_name,
            energy_spectrum_scale=self.energy_spectrum_scale,
        )
        sample_count = np.asarray(open_parameter_values).shape[0]
        spectra = convert_energy_spectra_to_channel_space_and_pileup(
            energy_result.spectra[self.method_name],
            calibration_offset=a,
            calibration_linear=b,
            calibration_quadratic=c,
            real_times=np.full((sample_count,), self.real_time, dtype=np.float64),
            live_times=np.full((sample_count,), self.live_time, dtype=np.float64),
            fudge_factors=np.full((sample_count,), self.fudge_factor, dtype=np.float64),
            normalization_factors=r,
        )
        lengths = np.full((sample_count,), spectra.shape[1], dtype=np.int32)
        return SimulationBatchResult(spectra={self.method_name: spectra}, spectra_lengths={self.method_name: lengths})

    def close(self):
        close = getattr(self.base, "close", None)
        if callable(close):
            close()


class SIMNRAChannelPileupSimulator:
    def __init__(self, input_spec, method_name, real_time, live_time, fudge_factor):
        self.input_spec = input_spec
        self.method_name = method_name
        self.real_time = float(real_time)
        self.live_time = float(live_time)
        self.fudge_factor = float(fudge_factor)
        self.base = SIMNRABatchSimulator(input_spec)
        self.method_names = [method_name]

    def simulate_batch(self, open_parameter_values, *, fixed_parameter_overrides=None, nthreads=1):
        channel_result = self.base.simulate_batch(
            open_parameter_values,
            fixed_parameter_overrides=fixed_parameter_overrides,
            nthreads=nthreads,
        )
        sample_count = np.asarray(open_parameter_values).shape[0]
        spectra = apply_channel_space_pileup(
            channel_result.spectra[self.method_name],
            real_times=np.full((sample_count,), self.real_time, dtype=np.float64),
            live_times=np.full((sample_count,), self.live_time, dtype=np.float64),
            fudge_factors=np.full((sample_count,), self.fudge_factor, dtype=np.float64),
        )
        lengths = np.full((sample_count,), spectra.shape[1], dtype=np.int32)
        return SimulationBatchResult(spectra={self.method_name: spectra}, spectra_lengths={self.method_name: lengths})

    def close(self):
        close = getattr(self.base, "close", None)
        if callable(close):
            close()

print("Fitting sample count:", fit_sample_count)
print("Initial guess shape:", initial_guess.shape)
print("Observed channel+pileup spectra shape:", observed_channel_pileup.shape)

In [ ]:
surrogate_simulator = SurrogateChannelPileupSimulator(
    input_spec=reference_dataset.input_spec,
    schema=schema,
    model=model,
    input_transform=input_scaler,
    output_inverse_transform=output_transform,
    method_name=method_name,
    real_time=real_time,
    live_time=live_time,
    fudge_factor=fudge_factor,
    energy_spectrum_scale=energy_spectrum_scale,
)

surrogate_fit_result = fit_open_parameters(
    surrogate_simulator,
    reference_dataset.input_spec,
    observed_spectra,
    observed_lengths=observed_lengths,
    initial_open_parameter_values=initial_guess,
    algo=surrogate_fit_algo,
    maxevals=surrogate_fit_maxevals,
    rel_tol=0.0,
    seed=fit_seed,
)
surrogate_fit_params = surrogate_fit_result.best_open_parameter_values
surrogate_eval_simulator = SurrogateChannelPileupSimulator(
    input_spec=reference_dataset.input_spec,
    schema=schema,
    model=model,
    input_transform=input_scaler,
    output_inverse_transform=output_transform,
    method_name=method_name,
    real_time=real_time,
    live_time=live_time,
    fudge_factor=fudge_factor,
    energy_spectrum_scale=energy_spectrum_scale,
)
surrogate_fit_chi2 = calculate_chi2_batch(
    surrogate_eval_simulator,
    surrogate_fit_params,
    observed_spectra,
    observed_lengths=observed_lengths,
).total

relative_param_error = np.abs(surrogate_fit_params - x_fit_true) / np.maximum(np.abs(x_fit_true), 1e-12)
print("Surrogate fit mean chi2:", float(np.mean(surrogate_fit_chi2)))
print("Surrogate fit median chi2:", float(np.median(surrogate_fit_chi2)))
print("Surrogate fit mean relative parameter error:", float(np.mean(relative_param_error)))
print("Surrogate fit results:")
for sample in surrogate_fit_result.samples:
    print(sample)

In [ ]:
try:
    simnra_simulator = SIMNRAChannelPileupSimulator(
        reference_dataset.input_spec,
        method_name=method_name,
        real_time=real_time,
        live_time=live_time,
        fudge_factor=fudge_factor,
    )
    simnra_fit_result = fit_open_parameters(
        simnra_simulator,
        reference_dataset.input_spec,
        observed_spectra,
        observed_lengths=observed_lengths,
        initial_open_parameter_values=initial_guess,
        algo=simnra_fit_algo,
        maxevals=simnra_fit_maxevals,
        rel_tol=0.0,
        seed=fit_seed,
        nthreads=simnra_fit_threads,
    )
    simnra_fit_params = simnra_fit_result.best_open_parameter_values
    simnra_eval_simulator = SIMNRAChannelPileupSimulator(
        reference_dataset.input_spec,
        method_name=method_name,
        real_time=real_time,
        live_time=live_time,
        fudge_factor=fudge_factor,
    )
    simnra_fit_chi2 = calculate_chi2_batch(
        simnra_eval_simulator,
        simnra_fit_params,
        observed_spectra,
        observed_lengths=observed_lengths,
        nthreads=simnra_fit_threads,
    ).total
    simnra_eval_simulator.close()
    simnra_relative_param_error = np.abs(simnra_fit_params - x_fit_true) / np.maximum(np.abs(x_fit_true), 1e-12)
    print("SIMNRA fit mean chi2:", float(np.mean(simnra_fit_chi2)))
    print("SIMNRA fit median chi2:", float(np.median(simnra_fit_chi2)))
    print("SIMNRA fit mean relative parameter error:", float(np.mean(simnra_relative_param_error)))
    print("SIMNRA fit results:")
    for sample in simnra_fit_result.samples:
        print(sample)
except Exception as exc:
    print("SIMNRA fitting skipped or failed:", exc)